## BPE vs Word Piece

Analyze vocabulary behavior by programmatically implementing and comparing subword tokenization schemes (BPE vs. WordPiece) and tracking corpus perplexity metrics.

## Setup

In [1]:
import collections
import math
import re

# 1. Define a toy corpus (frequencies of words)
corpus_words = {
    "low": 5,
    "lower": 2,
    "newest": 6,
    "widest": 3,
    "smartest": 2,
    "lowland": 2
}

# Pre-tokenize: split words into characters and append an End-Of-Word (</w>) token
def build_vocab(corpus):
    vocab = collections.Counter()
    for word, freq in corpus.items():
        # Space-separated characters
        chars = " ".join(list(word)) + " </w>"
        vocab[chars] = freq
    return vocab

initial_vocab = build_vocab(corpus_words)
print("Initial Vocabulary:\n", initial_vocab)

Initial Vocabulary:
 Counter({'n e w e s t </w>': 6, 'l o w </w>': 5, 'w i d e s t </w>': 3, 'l o w e r </w>': 2, 's m a r t e s t </w>': 2, 'l o w l a n d </w>': 2})


## Implementation

### BPE

In [2]:
def get_stats(vocab):
    """Count frequency of adjacent symbol pairs."""
    pairs = collections.Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq
    return pairs


def merge_vocab(pair, v_in):
    """Merge the specified pair in all words of the vocabulary."""
    v_out = {}
    bigram = re.escape(" ".join(pair))
    p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
    for word in v_in:
        w_out = p.sub("".join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out


def train_bpe(vocab, num_merges):
    bpe_vocab = vocab.copy()
    bpe_merges = []

    for i in range(num_merges):
        pairs = get_stats(bpe_vocab)
        if not pairs:
            break
        # BPE selects the most frequent pair
        best_pair = max(pairs, key=lambda k: pairs[k])
        bpe_vocab = merge_vocab(best_pair, bpe_vocab)
        bpe_merges.append(best_pair)

    return bpe_vocab, bpe_merges


# Train BPE
num_merges = 10
bpe_final_vocab, bpe_merge_history = train_bpe(initial_vocab, num_merges)
print("\n--- BPE Merges ---")
for i, merge in enumerate(bpe_merge_history):
    print(f"Merge {i+1}: {merge[0]} + {merge[1]} -> {''.join(merge)}")


--- BPE Merges ---
Merge 1: e + s -> es
Merge 2: es + t -> est
Merge 3: est + </w> -> est</w>
Merge 4: l + o -> lo
Merge 5: lo + w -> low
Merge 6: n + e -> ne
Merge 7: ne + w -> new
Merge 8: new + est</w> -> newest</w>
Merge 9: low + </w> -> low</w>
Merge 10: w + i -> wi


### WordPiece

In [3]:
def get_wordpiece_stats(vocab):
    """Calculate WordPiece scores for adjacent pairs."""
    pairs = collections.Counter()
    symbol_freqs = collections.Counter()

    for word, freq in vocab.items():
        symbols = word.split()
        for sym in symbols:
            symbol_freqs[sym] += freq
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i + 1]] += freq

    # Calculate scores: count(ab) / (count(a) * count(b))
    pair_scores = {}
    for pair, count in pairs.items():
        score = count / (symbol_freqs[pair[0]] * symbol_freqs[pair[1]])
        pair_scores[pair] = score

    return pair_scores


def train_wordpiece(vocab, num_merges):
    wp_vocab = vocab.copy()
    wp_merges = []

    for i in range(num_merges):
        scores = get_wordpiece_stats(wp_vocab)
        if not scores:
            break
        # WordPiece selects the pair with the highest score
        best_pair = max(scores, key=lambda k: scores[k])
        wp_vocab = merge_vocab(best_pair, wp_vocab)
        wp_merges.append(best_pair)

    return wp_vocab, wp_merges


# Train WordPiece
wp_final_vocab, wp_merge_history = train_wordpiece(initial_vocab, num_merges)
print("\n--- WordPiece Merges ---")
for i, merge in enumerate(wp_merge_history):
    print(f"Merge {i+1}: {merge[0]} + {merge[1]} -> {''.join(merge)}")


--- WordPiece Merges ---
Merge 1: m + a -> ma
Merge 2: ma + r -> mar
Merge 3: i + d -> id
Merge 4: a + n -> an
Merge 5: an + d -> and
Merge 6: l + o -> lo
Merge 7: l + and -> land
Merge 8: s + mar -> smar
Merge 9: s + t -> st
Merge 10: smar + t -> smart


## Evaluation

In [4]:
def calculate_metrics(tokenized_vocab):
    """Calculates Unigram Perplexity and average tokens per word (fertility)."""
    total_words = sum(tokenized_vocab.values())
    token_freqs = collections.Counter()
    total_tokens = 0

    # Count total tokens and individual token frequencies
    for word, freq in tokenized_vocab.items():
        tokens = word.split()
        total_tokens += len(tokens) * freq
        for token in tokens:
            token_freqs[token] += freq

    # Calculate Unigram Probability for each token
    token_probs = {token: count / total_tokens for token, count in token_freqs.items()}

    # Calculate Perplexity: 2^(- (1/N) * sum(log2(P(t_i))))
    log_prob_sum = 0
    for word, freq in tokenized_vocab.items():
        for token in word.split():
            log_prob_sum += math.log2(token_probs[token]) * freq

    perplexity = 2 ** (-log_prob_sum / total_tokens)
    fertility = total_tokens / total_words

    return perplexity, fertility


bpe_ppl, bpe_fertility = calculate_metrics(bpe_final_vocab)
wp_ppl, wp_fertility = calculate_metrics(wp_final_vocab)

print("\n=== Tokenization Metrics Comparison ===")
print(f"Metric\t\t\tBPE\t\tWordPiece")
print(f"-------------------------------------------------")
print(f"Tokens per word\t\t{bpe_fertility:.3f}\t\t{wp_fertility:.3f}")
print(f"Unigram Perplexity\t{bpe_ppl:.3f}\t\t{wp_ppl:.3f}")


=== Tokenization Metrics Comparison ===
Metric			BPE		WordPiece
-------------------------------------------------
Tokens per word		2.600		4.600
Unigram Perplexity	13.879		7.370
